In [2]:
# ============================================================
# PRODUCT RECOMMENDATION SYSTEM
# COLLABORATIVE FILTERING
# ============================================================

# ======================
# IMPORT LIBRARIES
# ======================

import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

# ======================
# LOAD DATASET
# ======================

df = pd.read_csv("eda_final_superstore.csv")

# ======================
# CREATE CUSTOMER-PRODUCT MATRIX
# ======================

customer_product_matrix = pd.pivot_table(
    df,
    index='Customer Name',
    columns='Product Name',
    values='Sales',
    aggfunc='sum',
    fill_value=0
)

# ======================
# COMPUTE COSINE SIMILARITY
# ======================

similarity_matrix = cosine_similarity(customer_product_matrix)

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=customer_product_matrix.index,
    columns=customer_product_matrix.index
)

# ======================
# RECOMMENDATION FUNCTION
# ======================

def recommend_products(customer_name, top_n=5):

    # Find similar customers
    similar_customers = similarity_df[customer_name].sort_values(
        ascending=False
    )[1:6]

    similar_customer_names = similar_customers.index

    # Products already purchased
    purchased_products = customer_product_matrix.loc[
        customer_name
    ]

    purchased_products = purchased_products[
        purchased_products > 0
    ].index.tolist()

    # Products from similar customers
    recommended_products = []

    for sim_customer in similar_customer_names:

        sim_products = customer_product_matrix.loc[
            sim_customer
        ]

        sim_products = sim_products[
            sim_products > 0
        ].index.tolist()

        for product in sim_products:

            if product not in purchased_products:
                recommended_products.append(product)

    # Remove duplicates
    recommended_products = list(dict.fromkeys(
        recommended_products
    ))

    return recommended_products[:top_n]

# ======================
# TEST RECOMMENDATIONS
# ======================

sample_customer = customer_product_matrix.index[0]

print("\nSample Customer:")
print(sample_customer)

recommendations = recommend_products(sample_customer)

print("\nRecommended Products:")

for i, product in enumerate(recommendations, start=1):
    print(f"{i}. {product}")

# ======================
# SAVE SIMILARITY MATRIX
# ======================

similarity_df.to_csv(
    "customer_similarity_matrix.csv"
)

print("\nRecommendation System Completed Successfully.")


Sample Customer:
Aaron Bergman

Recommended Products:
1. Acco D-Ring Binder w/DublLock
2. Avery 482
3. Chromcraft 48" x 96" Racetrack Double Pedestal Table
4. Global Highback Leather Tilter in Burgundy
5. Imation 32GB Pocket Pro USB 3.0 Flash Drive - 32 GB - Black - 1 P ...

Recommendation System Completed Successfully.
